In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
import torch
import os
import pandas as pd
import re
import emoji

from dotenv import load_dotenv
load_dotenv()  

In [ ]:
data_path = os.getenv('DATA_PATH')
comments = pd.read_csv(f'{data_path}/all_labeled_comments.csv')
comments_train_data = pd.read_csv(f"{data_path}/labeled_comments.csv")


def preprocessing(comment):
    """
    Preprocesses a given comment by performing the following steps:
    1. Replaces all emojis with a space.
    2. Removes URLs.
    3. Removes all non-word characters (punctuation).
    4. Removes all digits.

    Parameters:
    comment (str): The input comment to be preprocessed.

    Returns:
    str: The preprocessed comment.
    """
    comment = emoji.replace_emoji(comment, replace=" ")
    comment = re.sub(r'https?://\S+|www\.\S+', ' ', comment)
    comment = re.sub(r'[^\w\s]', ' ', comment)
    comment = re.sub(r'[^\w\s\n]', ' ', comment)
    # comment = re.sub(r'\d+', ' ', comment)
    return comment

df = comments[comments['delivery']=='False'].sample(1000)
df['preprocessed_comment'] = df['description'].apply(preprocessing)

### finetuned parsbert sanppfood

In [ ]:
# load model
model_path = os.getenv('MODELS')
save_directory = f"{model_path}/fine_tuned_parsbert_snappfood_for_bad_delivery_detection"

tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() 

In [ ]:
batch_size = 32

def predict_labels(texts, batch_size=64):
    all_preds = []

    with torch.no_grad():  
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]  
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()} 

            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, axis=1).cpu().numpy()
            all_preds.extend(preds)

    return all_preds


df["predicted_label"] = predict_labels(df["preprocessed_comment"].tolist(), batch_size=batch_size)

label_map = {0: "Not a complaint", 1: "Bad delivery complaint"}
df["predicted_label"] = df["predicted_label"].map(label_map)

In [ ]:
# detection by some specific words

delay_keywords = ["دیر رسید", "دیر به دستم رسید", "تحویل دیر", "تاخیر در ارسال", "خیلی طول کشید", "ارسال نشد", "تحویل"]
wrong_item_keywords = ["اشتباه فرستادید", "کالای اشتباه", "محصول اشتباه", "چیزی که سفارش دادم نبود", "فرق داشت", "ارسال"]
packaging_keywords = ["بسته بندی خراب", "بسته بندی بد", "بسته بندی نداشت", "بسته بندی خوبی نداشت", "بسته بندی مشکل", "بسته بندی ضعیف", "پلاستیک پاره"]

all_keywords = delay_keywords + wrong_item_keywords + packaging_keywords

def label_comment(description):
    for keyword in all_keywords:
        if keyword in description:
            return True
    return False

df["delivery_issue"] = df["preprocessed_comment"].astype(str).apply(label_comment)

df[df['delivery_issue']==True]['description'].values

In [ ]:
for i in df[df['predicted_label']=='Bad delivery complaint'].values:
    print(i[1], i[3], i[7], i[9])

In [ ]:
#  test
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = logits.argmax().item()
    return "Bad Delivery" if predicted_class == 1 else "Not Bad Delivery"

new_comment = "خیلی بد بود"
print(predict(new_comment))

### finetuned bert-fa-base-uncased 

In [ ]:
# load model
model_path = os.getenv('MODELS')
save_directory = f"{model_path}/fine_tuned_bert-fa-base-uncased_for_bad_delivery_detection/v2"

tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() 

In [ ]:
batch_size = 32

def predict_labels(texts, batch_size=64):
    all_preds = []

    with torch.no_grad():  
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]  
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()} 

            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, axis=1).cpu().numpy()
            all_preds.extend(preds)

    return all_preds


df["predicted_label"] = predict_labels(df["preprocessed_comment"].tolist(), batch_size=batch_size)

label_map = {0: "Not a complaint", 1: "Bad delivery complaint"}
df["predicted_label"] = df["predicted_label"].map(label_map)